# Modelo estatístico: taxa de desocupação

Regressão linear para estimar a taxa de 2025 pela taxa de 2024 da mesma UF. Avaliação leave-one-out: cada UF é prevista por um modelo ajustado apenas nas outras 11. Compara-se com a hipótese de repetir 2024. Recorte de 12 UFs selecionadas, não representativo. Para executar no Colab, use Executar tudo e selecione o CSV quando solicitado. Não é previsão para 2026.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path

caminho = Path('dados/desocupacao_ibge_2024_2025.csv')
if not caminho.exists():
    from google.colab import files
    print('Selecione o arquivo desocupacao_ibge_2024_2025.csv')
    files.upload()
    caminho = Path('desocupacao_ibge_2024_2025.csv')

df = pd.read_csv(caminho)
assert not df.duplicated(['uf', 'ano']).any()
tabela = df.pivot(index='uf', columns='ano', values='taxa_desocupacao_pct').dropna(subset=[2024, 2025])
X = tabela[[2024]]
y = tabela[2025]
previsto = cross_val_predict(LinearRegression(), X, y, cv=LeaveOneOut())
mae = mean_absolute_error(y, previsto)
rmse = np.sqrt(mean_squared_error(y, previsto))
mae_referencia = mean_absolute_error(y, X[2024])
print(f'UFs avaliadas: {len(tabela)}')
print(f'MAE do modelo: {mae:.3f} ponto percentual')
print(f'RMSE do modelo: {rmse:.3f} ponto percentual')
print(f'MAE da referência (2025 igual a 2024): {mae_referencia:.3f} ponto percentual')
resultado = pd.DataFrame({'UF': tabela.index, 'observado_2024': X[2024].values, 'observado_2025': y.values, 'estimado_2025_validacao': previsto})
resultado['erro_absoluto_pp'] = abs(resultado['observado_2025'] - resultado['estimado_2025_validacao'])
print('\nEstimativas fora da amostra (uma UF de cada vez):')
print(resultado.round(3).to_string(index=False))


UFs avaliadas: 12
MAE do modelo: 0.240 ponto percentual
RMSE do modelo: 0.326 ponto percentual
MAE da referência (2025 igual a 2024): 0.542 ponto percentual

Estimativas fora da amostra (uma UF de cada vez):
                 UF  observado_2024  observado_2025  estimado_2025_validacao  erro_absoluto_pp
              Amapá             8.3             7.9                    7.894             0.006
           Amazonas             8.4             8.4                    7.887             0.513
              Ceará             7.0             6.5                    6.528             0.028
     Espírito Santo             3.9             3.3                    3.249             0.051
           Maranhão             7.1             6.8                    6.607             0.193
        Mato Grosso             2.6             2.2                    1.762             0.438
 Mato Grosso do Sul             3.9             3.0                    3.303             0.303
       Minas Gerais             

## Interpretação e limites

O erro médio absoluto é cerca de 0,24 ponto percentual, contra 0,54 na referência. São apenas 12 UFs selecionadas por estarem em ambas as notícias de menores taxas, com dois anos e valores arredondados. A avaliação pode ser otimista para outras UFs ou anos e não estima incerteza; a associação não estabelece causa. O uso em decisões exigiria dados mais amplos e validação temporal externa.